# Beacon - Correlation-Sensitive Next-Basket Recommendation

Paper: [Correlation-Sensitive Next-Basket Recommendation](https://www.ijcai.org/proceedings/2019/0389.pdf) (IJCAI'19)

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
cd ..

/home/jupyter/work/resources/hse-masters-thesis-2026


In [3]:
import torch
import os
import sys
import pandas as pd
import numpy as np

# project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
# if project_root not in sys.path:
#     sys.path.insert(0, project_root)
# os.chdir(project_root)

from tecd_retail_recsys.models.beacon import BeaconTrainer
from tecd_retail_recsys.data import DataPreprocessor
from tecd_retail_recsys.metrics import calculate_metrics

/home/jupyter/.local/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [6]:
dp = DataPreprocessor(
    day_begin=1082, 
    day_end=1308, 
    val_days=20, 
    test_days=20, 
    min_user_interactions=1, 
    min_item_interactions=20
)

train_df, val_df, test_df = dp.preprocess()

Starting data preprocessing...
Loading events from t_ecd_small_partial/dataset/small/retail/events
Loaded 267,043,972 total events
Loading items data from t_ecd_small_partial/dataset/small/retail/items.pq
Loaded 250,171 items with features: ['item_id', 'item_brand_id', 'item_category', 'item_subcategory', 'item_price', 'item_embedding']
Merged item features. Data shape: (267043972, 12)


IOStream.flush timed out


Filtered to 3,852,145 events with action_type='added_to_cart'
After filtering (min_user_interactions=1, min_item_interactions=20): 3,327,057 events, 84,830 users, 32,095 items
Created mappings: 84830 users, 32095 items
Temporal split - Train: days < 1269 (922,368 events), Val: days 1269-1288 (232,900 events), Test: days >= 1289 (227,959 events)
Users in each part (train, val, test) - 7422


In [5]:
def group_by_baskets(df, time_window_hours=2, min_basket_size=2):
    """
    Group interactions into baskets by user using time windows.
    """
    if not pd.api.types.is_datetime64_any_dtype(df['timestamp']):
        df = df.copy()
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s')
    
    df = df.sort_values(['user_id', 'timestamp']).reset_index(drop=True)
    
    user_baskets = {}
    
    for user_id, group in df.groupby('user_id'):
        group = group.sort_values('timestamp')
        timestamps = group['timestamp'].values
        items = group['item_id'].values
        
        time_delta = pd.to_timedelta(time_window_hours, unit='h')
        
        baskets = []
        current_basket = [items[0]]
        
        for i in range(1, len(items)):
            gap = pd.Timestamp(timestamps[i]) - pd.Timestamp(timestamps[i - 1])
            if gap <= time_delta:
                current_basket.append(items[i])
            else:
                if len(current_basket) >= min_basket_size:
                    seen = set()
                    deduped = []
                    for item in current_basket:
                        if item not in seen:
                            seen.add(item)
                            deduped.append(item)
                    baskets.append(deduped)
                current_basket = [items[i]]

        if len(current_basket) >= min_basket_size:
            seen = set()
            deduped = []
            for item in current_basket:
                if item not in seen:
                    seen.add(item)
                    deduped.append(item)
            baskets.append(deduped)
        
        if baskets:
            user_baskets[user_id] = baskets
    
    user_ids = list(user_baskets.keys())
    return user_baskets, user_ids


train_user_baskets, train_user_ids = group_by_baskets(train_df, time_window_hours=2, min_basket_size=2)
val_user_baskets, val_user_ids = group_by_baskets(val_df, time_window_hours=2, min_basket_size=2)

basket_sizes = [len(b) for baskets in train_user_baskets.values() for b in baskets]
seq_lengths = [len(baskets) for baskets in train_user_baskets.values()]

print(f"Users:                 {len(train_user_baskets)}")
print(f"Total baskets:         {len(basket_sizes)}")
print(f"Basket size:           median={np.median(basket_sizes):.0f}, "
      f"mean={np.mean(basket_sizes):.1f}, "
      f"max={np.max(basket_sizes)}")
print(f"Baskets per user:      median={np.median(seq_lengths):.0f}, "
      f"mean={np.mean(seq_lengths):.1f}, "
      f"max={np.max(seq_lengths)}")
print(f"Users with ≥2 baskets: {sum(1 for s in seq_lengths if s >= 2)} "
      f"({sum(1 for s in seq_lengths if s >= 2)/len(seq_lengths)*100:.0f}%)")


Users:                 7161
Total baskets:         119230
Basket size:           median=4, mean=6.4, max=117
Baskets per user:      median=11, mean=16.6, max=258
Users with ≥2 baskets: 6674 (93%)


In [ ]:
trainer = BeaconTrainer(
    train_user_baskets,
    embed_dim=128,
    hidden_dim=128,
    n_lstm_layers=2,
    alpha=0.2,             # баланс correlation vs sequential
    dropout=0.1,
    corr_top_n=5,           # top-N в матрице корреляций
    corr_mu=0.85,           # diffusion
    lr=1e-3,
    weight_decay=1e-5,
    batch_size=2048,
    epochs=10,
    holdout_ratio=0.15,
    pos_weight=50,
    eval_k=100,
    max_seq_len=20,
    device="auto",
    verbose=True,
)


Items:            30303
User sequences:   6674
Train sequences:  5673
Eval sequences:   1001
Device:           cuda
Building correlation matrix...
Items:            30303
User sequences:   6674
Train sequences:  5673
Eval sequences:   1001
Train samples:    95245
Eval samples:     16824
Device:           cuda
Model params:     8,082,881


In [10]:
trainer.train()

Epoch   1/10 │ loss=0.2569 │ NDCG@100=0.0753 │ Recall@100=0.2410
Epoch   2/10 │ loss=0.0518 │ NDCG@100=0.1263 │ Recall@100=0.3081
Epoch   3/10 │ loss=0.0501 │ NDCG@100=0.1391 │ Recall@100=0.3117
Epoch   4/10 │ loss=0.0497 │ NDCG@100=0.1404 │ Recall@100=0.3130
Epoch   5/10 │ loss=0.0494 │ NDCG@100=0.1413 │ Recall@100=0.3130
Epoch   6/10 │ loss=0.0492 │ NDCG@100=0.1419 │ Recall@100=0.3130
Epoch   7/10 │ loss=0.0491 │ NDCG@100=0.1406 │ Recall@100=0.3124
Epoch   8/10 │ loss=0.0490 │ NDCG@100=0.1405 │ Recall@100=0.3128
Epoch   9/10 │ loss=0.0489 │ NDCG@100=0.1408 │ Recall@100=0.3128
Epoch  10/10 │ loss=0.0489 │ NDCG@100=0.1409 │ Recall@100=0.3127


In [11]:
torch.save({
    "model_state_dict": trainer.model.state_dict(),
    "optimizer_state_dict": trainer.optimizer.state_dict(),
    "scheduler_state_dict": trainer.scheduler.state_dict(),
    "item2idx": trainer.item2idx,
    "idx2item": trainer.idx2item,
    "num_items": trainer.num_items,
    "eval_k": trainer.eval_k,
    "batch_size": trainer.batch_size,
    "max_seq_len": trainer.max_seq_len,
    "max_basket_size": trainer.max_basket_size,
    "pos_weight": trainer.pos_weight,
    "model_config": {
        "num_items": trainer.model.num_items,
        "embed_dim": trainer.model.embed_dim,
        "hidden_dim": trainer.model.hidden_dim,
        "n_lstm_layers": trainer.model.sequence_encoder.lstm.num_layers,
        "alpha": trainer.model.predictor.alpha,
        "dropout": trainer.model.dropout.p,
    },
}, "models/beacon/best_model.pt")

In [ ]:
metrics = trainer.evaluate(val_user_baskets)  # последняя корзина - тестовая

Sequences evaluated: 24027
NDCG@100:         0.0892
Recall@100:       0.1847


In [15]:
recs = {}
from tqdm import tqdm
for user_id in tqdm(train_user_baskets.keys()):
    user_recs = trainer.predict(train_user_baskets[user_id], top_k=100)
    recs[user_id] = [item_id for item_id, score in user_recs]

100%|██████████| 7161/7161 [01:07<00:00, 106.82it/s]


In [ ]:
recs_df = pd.DataFrame(recs.items(), columns=['user_id', 'beacon_recs'])
recs_df.to_csv('models/beacon/recs_df.csv', index=False)

In [12]:
joined = dp.get_grouped_data(train_df, val_df, test_df)

joined = joined.merge(recs_df, on='user_id')
beacon_metrics = calculate_metrics(
    joined,
    train_col='train_interactions',
    model_preds='beacon_recs',
    gt_col='val_interactions',
    verbose=True)

[Metrics debug] resolved gt_col='val_interactions' item_id_index=0
[Metrics debug] ratings_true shape: (19124, 3) ratings_pred shape: (62200, 3)
  ratings_true dtypes: {'user_id': dtype('int64'), 'item_id': dtype('int64')}
  ratings_pred dtypes: {'user_id': dtype('int64'), 'item_id': dtype('int64')}
  user_id=11 gt_count=22 pred_count=100 overlap=0
    [ID spaces] gt sample=[489, 502, 750, 2781, 4010] range=[489, 29937] | rec sample=[362, 492, 734, 1223, 1463] range=[362, 30418]
  user_id=201 gt_count=14 pred_count=100 overlap=0
    [ID spaces] gt sample=[13, 6651, 7031, 10009, 16412] range=[13, 27697] | rec sample=[108, 497, 863, 1089, 1936] range=[108, 30314]
  user_id=247 gt_count=16 pred_count=100 overlap=0
    [ID spaces] gt sample=[154, 432, 5763, 10652, 11437] range=[154, 29437] | rec sample=[40, 73, 362, 805, 1223] range=[40, 30418]

At k=10:
  MAP@10       = 0.0001
  NDCG@10      = 0.0004
  Precision@10 = 0.0005
  Recall@10    = 0.0001

At k=100:
  MAP@100       = 0.0001
  NDC

`В целом удается рекомендовать одну следующую корзину: NDCG@100 = 0.0892, но порекомендовать сразу 100 айтемов (по сути - сразу несколько следующих корзин) не получается (NDCG@100 около нуля).`

In [17]:
from tecd_retail_recsys.utils import get_avg_recs_price, get_item_to_price

item_to_price = get_item_to_price(dp)
avg_recs_price = get_avg_recs_price(joined, item_to_price, 'beacon_recs')
print(f'Средняя цена рекомендаций модели Beacon: {avg_recs_price:.2f}')

Средняя цена рекомендаций модели Beacon: -4.10
